Lets begin by reading data/seq.txt into a list of sequences we can use

In [1]:

seqs = []
with open('data/seq.txt') as f:
    for line in f:
        line = line.strip()
        # Strip the last 3 characters 'DNA'
        seqs.append(line[:-3])
        seqs.append(line.strip())

# print(seqs)

Lets begin by creating the comparison matrix between all sequences, using BLOOM62

In [3]:
from Bio.Align import PairwiseAligner
from Bio.Align import substitution_matrices
import numpy as np

def compute_smith_waterman_bioaligner(seq1, seq2):
    """
    Computes the Smith-Waterman alignment score for two sequences using PairwiseAligner.
    """
    # Load BLOSUM62 substitution matrix
    matrix = substitution_matrices.load("BLOSUM62")

    # Initialize the aligner
    aligner = PairwiseAligner()
    aligner.substitution_matrix = matrix
    aligner.open_gap_score = -4
    aligner.extend_gap_score = -1

    # Perform the alignment
    score = aligner.score(seq1, seq2)
    return score


def compute_comparison_matrix(sequences):
    """
    Computes a symmetric comparison matrix for a list of sequences.
    """
    n = len(sequences)
    comparison_matrix = np.zeros((n, n))

    print("Starting comparison matrix computation...")
    for i in range(n):
        # print(f"Processing sequence {i+1}/{n}")
        for j in range(i, n):
            score = compute_smith_waterman_bioaligner(sequences[i], sequences[j])
            comparison_matrix[i, j] = score
            comparison_matrix[j, i] = score  # Symmetry

    print("Finished computation.")
    return comparison_matrix

Now lets begin by creating a comparison matrix between all the sequences and displaying it nicely

In [5]:
original_matrix = compute_comparison_matrix(seqs)
print(original_matrix)

Starting comparison matrix computation...
Finished computation.
[[2854. 2848. 2819. ... 2254. 2342. 2336.]
 [2848. 2870. 2813. ... 2275. 2336. 2358.]
 [2819. 2813. 2849. ... 2233. 2308. 2302.]
 ...
 [2254. 2275. 2233. ... 2737. 1796. 1817.]
 [2342. 2336. 2308. ... 1796. 2787. 2781.]
 [2336. 2358. 2302. ... 1817. 2781. 2803.]]


Note how a higher allignment score means the sequences are more closely matched. 
Now, lets build out the tree, beginning with a matrix of all sequences, and gradually "building up", keeping track of the merges that we made.

In [15]:
import numpy as np
from copy import deepcopy

# Initialize clusters and Newick strings
current_clusters = [[i] for i in range(len(seqs))]
newick_nodes = [str(i) for i in range(len(seqs))]

def compute_average_score(cluster1, cluster2, original_matrix, linkage='average'):
    scores = [original_matrix[elem1, elem2] for elem1 in cluster1 for elem2 in cluster2]
    if linkage == 'average':
        return np.mean(scores)
    elif linkage == 'complete':
        return np.max(scores)
    elif linkage == 'single':
        return np.min(scores)


while len(current_clusters) > 1:
    # Find the highest normalized score
    max_score = -np.inf
    to_merge = (None, None)
    
    for i in range(len(current_clusters)):
        for j in range(i + 1, len(current_clusters)):
            # Use average linkage
            score = compute_average_score(current_clusters[i], current_clusters[j], original_matrix)
            if score > max_score:
                max_score = score
                to_merge = (i, j)

    # Merge the clusters with the highest score
    cluster1_idx, cluster2_idx = to_merge
    cluster1 = current_clusters[cluster1_idx]
    cluster2 = current_clusters[cluster2_idx]
    new_cluster = cluster1 + cluster2

    # Create new Newick node with distances
    new_newick = f"({newick_nodes[cluster1_idx]},{newick_nodes[cluster2_idx]})"
    
    # Update newick_nodes list
    newick_nodes = (
        newick_nodes[:cluster1_idx] + 
        [new_newick] + 
        newick_nodes[cluster1_idx + 1:cluster2_idx] + 
        newick_nodes[cluster2_idx + 1:]
    )
    
    # Note in the above, how the newick nodes indexes will match the current_clusters indexes

    # Update the clusters
    current_clusters = (
        current_clusters[:cluster1_idx] + 
        [new_cluster] + 
        current_clusters[cluster1_idx + 1:cluster2_idx] + 
        current_clusters[cluster2_idx + 1:]
    )

    # Print progress
    
    # print(f"Merged clusters {cluster1_idx} and {cluster2_idx} with score {max_score:.4f}")
    # print(f"Current clusters: {current_clusters}")

# Add final semicolon to complete Newick string
final_newick = newick_nodes[0] + ";"
print("\nFinal Newick string:", final_newick)


Final Newick string: ((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((0,1),(98,99)),(146,147)),((162,163),(178,179))),(130,131)),(66,67)),(2,3)),(194,195)),(52,53)),(34,35)),(50,51)),(18,19)),(82,83)),(4,5)),(86,87)),(20,21)),(196,197)),(228,229)),(166,167)),(210,211)),(214,215)),(70,71)),(226,227)),(134,135)),(180,181)),(56,57)),(164,165)),(230,231)),(100,101)),(10,11)),(152,153)),(234,235)),(212,213)),(24,25)),(118,119)),(116,117)),((154,155),(200,201))),(148,149)),(38,39)),(40,41)),(72,73)),(170,171)),(8,9)),(90,91)),(120,121)),(36,37)),(182,183)),(202,203)),(136,137)),(22,23)),(54,55)),(6,7)),(84,85)),(132,133)),(150,151)),(76,77)),(102,103)),(58,59)),(186,187)),(62,63)),(114,115)),(106,107)),(46,47)),(74,75)),(168,169)),(88,89)),(108,109)),(216,217)),(198,199)),(12,13)),(172,173)),(184,185)),(104,105)),(140,141)),(236,237)),(138,139)),(26,27)),(124,125)),(218,219)),(30,31)),(128,129)),(188,189)),

Now, let's display the tree nicely

In [16]:
from ete3 import Tree

tree = Tree(final_newick)
print(tree)


                                                                                                                                                                                                                                                                                                                                                                                                                  /-0
                                                                                                                                                                                                                                                                                                                                                                                                               /-|
                                                                                                                                                                                              